In [ ]:
import pandas as pd 
df = pd.read_csv("train.csv")
display(df.head());
print(df.shape)

In [ ]:
df.describe()

In [ ]:
df.drop("id", axis=1)

In [ ]:
df.dtypes

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# X = df.drop(columns=["Will_Buy_EV"])
# y = df["Will_Buy_EV"]

X = pd.get_dummies(df.drop(columns=["Will_Buy_EV", "id"]), drop_first=True)
y = df["Will_Buy_EV"].map({"Yes": 1, "No": 0})

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
pred = model.predict_proba(X_valid)[:, 1]
print(roc_auc_score(y_valid, pred))

* Now let's try to perform standardization here as 6 of the columns are categorical.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict_proba(X_valid)[:, 1]

print(roc_auc_score(y_valid, pred))

* Accuracy Improved from 0.90 to 0.93
## Let's try using XGBoost now

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = pd.get_dummies(df.drop(columns=["Will_Buy_EV", "id"]), drop_first=True)
y = df["Will_Buy_EV"].map({"Yes": 1, "No": 0})

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict_proba(X_valid)[:, 1]

print(roc_auc_score(y_valid, pred))

* Reached 0.90 Accuracy -> 0.93 (Using Standardization) -> 0.94 (Using XGBoost)
* Let's try CatBoost now :

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

model.fit(X_train, y_train)

pred = model.predict_proba(X_valid)[:, 1]

print(roc_auc_score(y_valid, pred))

* CatBoost reaches a 0.0008 lower accuracy than XGBoost, which is insignificant in hindsight.
* Before moving forward let's try plotting some graphs or subplots and check out how the data is distributed.

In [ ]:
%pip install seaborn
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=df, x="Will_Buy_EV", y="Age")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Will_Buy_EV", y="Annual_Income_USD")
plt.show()

In [ ]:
sns.boxplot(
    data=df,
    x="Will_Buy_EV",
    y="Charging_Stations_Near_Home"
)
plt.show()

* Plots really didn't tell me anything, the dataset is pretty evenly distributed apart from the fact that low-mid earning individuals don't prefer purchasing EV's.
* Let's try to get into more detail for this using some plots.

In [ ]:
pd.crosstab(
    pd.qcut(df["Annual_Income_USD"], 5),
    df["Will_Buy_EV"],
    normalize="index"
)

In [ ]:
df.groupby("Will_Buy_EV")["Annual_Income_USD"].mean()

* Income feels highly predictive of the final result now, it keeps increasing as we go up and seems the strongest feature in our dataframe.

In [ ]:
sns.boxplot(data=df, x="Will_Buy_EV", y="Annual_Income_USD")

In [ ]:
pd.crosstab(df["Subsidy_Available"], df["Will_Buy_EV"], normalize="index")

* We've just realized, subsidy available is a incredibly strong feature as well, maybe even stronger than Income.
* Let's try to build and train our better baseline model now

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1}) # just binary classification 
X = train.drop(columns=["Will_Buy_EV"])
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    verbose=50,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

predictions = model.predict_proba(X_valid)[:, 1]
score = roc_auc_score(y_valid, predictions)
print("ROC-AUC:", score)

* Okay 0.941 is good enough for my first proper baseline model, the leaderboard top is 0.946 so I'm far by 0.005.
* Let's make this submission and then try newer techniques later.

In [ ]:
# Preparing a submission.csv for submission now.
test_predictions = model.predict_proba(test)[:, 1]
submission = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(submission.shape)

* Let's run k-fold now on the training dataset and then check results and optimize the hyperparameters.


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})
X = train.drop(columns=["Will_Buy_EV"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), 1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=False,
        random_seed=42
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols
    )

    predictions = model.predict_proba(X_valid)[:, 1]
    score = roc_auc_score(y_valid, predictions)

    scores.append(score)

    print(f"Fold {fold}: {score:.6f}")

print("\nMean ROC-AUC:", sum(scores) / len(scores))
print("Std ROC-AUC:", pd.Series(scores).std())

* Final baseline model set at accuracy 0.941, let the hyperparameters tuning begin!

# Hyperparameter Tuning

## 1. Depth

| Depth | ROC-AUC |
|------:|--------:|
| 4     | 0.941000 |
| 6     | 0.941067 |
| 8     | **0.941125** |
| 10    | 0.941009 |
| 11    | 0.941096 |
| 12    | 0.940486 |

The ROC-AUC peaked at **depth = 8**, so we'll pick this as our depth hyperparameter.

---

## 2. Learning Rate

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 0.01     | 0.9397632329260461 |
| 0.03     | 0.9406599579823963 |
| 0.05     | 0.941125 |
| 0.1     |  **0.9412652935364746** |
| 0.2    | 0.9402401598427775 |

The ROC-AUC peaked at **Learning Rate = 0.1**, so we'll pick this as our learning rate hyperparameter.

---


## 3. L2 Regularization

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 1     | 0.9400649805975736 |
| 3     | **0.9412652935364746** |
| 5     | 0.9403550589882825 |
| 10     | 0.9405684202847963  |

The ROC-AUC peaked at **Learning Rate = 0.1**, so we'll pick this as our learning rate hyperparameter.

---

## 4. Iterations

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 300     | **0.9406698297215036** |
| 500     | 0.9399540407762492 |
| 800     | 0.9389406070076945  |
| 1200     | 0.937831279004862  |
| 2000     | 0.9358642051373034  |

The ROC-AUC peaked at **Iterations = 0.1**, so we'll pick this as our iterations hyperparameter.

---

## 5. Random Strength

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 1     | 0.9400649805975736 |
| 2     | 0.9401554965591212 |

The ROC-AUC peaked at **Random Strength = 2**, so we'll pick this as our Random Strength hyperparameter.

---

## 5. Bagging Temperature

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 0     | 0.9400649805975736 |
| 0.5   | 0.9400649805975736 |
| 1     | 0.9400649805975736 |
| 2     | 0.9400649805975736 |

The ROC-AUC peaked at **Bagging Temperature = 0**, so we'll pick this as our Bagging Temperature hyperparameter.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})
X = train.drop(columns = ["Will_Buy_EV"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

model = CatBoostClassifier(
    iterations = 300,
    learning_rate = 0.1,
    depth = 8,
    thread_count=-1,
    l2_leaf_reg = 3,
    verbose = 100,
    random_seed = 42,
    bagging_temperature=0,
    random_strength=2,
    early_stopping_rounds=100
)

model.fit(
    X_train,
    y_train,
    cat_features = cat_cols
)

predictions = model.predict_proba(X_valid)[:, 1]
score = roc_auc_score(y_valid, predictions)
print("ROC-AUC:", score)

* After testing on the tuned hyperparameters an accuracy of 0.941076 is achieved.
* Let's create a submission.csv and submit it for now.

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})

X = train.drop(columns=["Will_Buy_EV"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=8,
    thread_count=-1,
    l2_leaf_reg=3,
    verbose=100,
    random_seed=42,
    bagging_temperature=0,
    random_strength=2
)

model.fit(
    X,
    y,
    cat_features=cat_cols
)

test_predictions = model.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Submission shape:", submission.shape)